# 05: Explorando P/L, ROE e DY via yfinance

Três métricas básicas de valuation, para ter uma primeira ideia de preço e retorno de uma ação.

| Métrica | O que mede | Referência rápida |
|---|---|---|
| **P/L** (P/E) | Quantos anos de lucro o mercado paga pela ação | Menor = mais barato |
| **ROE** | Quanto a empresa lucra sobre o patrimônio dos sócios | Maior = mais eficiente |
| **DY** (Dividend Yield) | Quanto a ação paga de dividendo em % do preço | Maior = mais renda |

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

In [2]:
import yfinance as yf

TICKER = 'WEGE3.SA'
acao = yf.Ticker(TICKER)
info = acao.info

print(f'Empresa: {info.get("longName", TICKER)}')

Empresa: WEG S.A.


## Onde as métricas moram no yfinance

Todas as três vivem em `ticker.info`, um dicionário com ~100 campos calculados pelo Yahoo Finance.

In [3]:
# Campos brutos do info
campos = {
    'P/L':  ('trailingPE', 'Preço / Lucro por ação (últimos 12 meses)'),
    'ROE':  ('returnOnEquity', 'Lucro Líquido / Patrimônio Líquido'),
    'DY':   ('dividendYield', 'Dividendo anual / Preço atual'),
}

for nome, (chave, descricao) in campos.items():
    valor = info.get(chave)
    print(f'{nome:5}  chave: {chave:<20}  valor bruto: {valor}')
    print(f'       {descricao}')
    print()

P/L    chave: trailingPE            valor bruto: 32.34228
       Preço / Lucro por ação (últimos 12 meses)

ROE    chave: returnOnEquity        valor bruto: 0.29878
       Lucro Líquido / Patrimônio Líquido

DY     chave: dividendYield         valor bruto: 3.22
       Dividendo anual / Preço atual



In [4]:
# Formatados para leitura
pl  = info.get('trailingPE')
roe = info.get('returnOnEquity')
dy  = info.get('dividendYield')

print(f'=== {info.get("longName", TICKER)} ===')
print(f'Preço atual:    R$ {info.get("currentPrice", "N/A")}')
print()
print(f'P/L:   {pl:.1f}x'              if pl  else 'P/L:   N/A')
print(f'ROE:   {roe*100:.1f}%'          if roe else 'ROE:   N/A')
print(f'DY:    {dy*100:.2f}% ao ano'    if dy  else 'DY:    N/A (empresa não paga dividendos ou dado ausente)')

=== WEG S.A. ===
Preço atual:    R$ 48.19

P/L:   32.3x
ROE:   29.9%
DY:    322.00% ao ano


## Explorando o que mais tem no `info`

Útil para saber o que está disponível para enriquecer a tool.

In [5]:
# Campos interessantes para análise fundamentalista
campos_uteis = [
    'currentPrice', 'marketCap',
    'trailingPE', 'forwardPE', 'priceToBook',
    'returnOnEquity', 'returnOnAssets',
    'dividendYield', 'trailingAnnualDividendYield',
    'debtToEquity', 'currentRatio',
    'revenueGrowth', 'earningsGrowth',
    'grossMargins', 'operatingMargins', 'profitMargins',
]

print(f'Campos disponíveis em {TICKER}:')
print(f'{"Campo":<40} {"Valor"}')
print('-' * 60)
for campo in campos_uteis:
    valor = info.get(campo)
    if valor is not None:
        if isinstance(valor, float) and valor < 10:
            print(f'{campo:<40} {valor:.4f}')
        elif isinstance(valor, float):
            print(f'{campo:<40} {valor:,.0f}')
        else:
            print(f'{campo:<40} {valor}')
    else:
        print(f'{campo:<40} -')

Campos disponíveis em WEGE3.SA:
Campo                                    Valor
------------------------------------------------------------
currentPrice                             48
marketCap                                202197024768
trailingPE                               32
forwardPE                                26
priceToBook                              10
returnOnEquity                           0.2988
returnOnAssets                           0.1130
dividendYield                            3.2200
trailingAnnualDividendYield              0.0415
debtToEquity                             21
currentRatio                             1.5390
revenueGrowth                            0.0570
earningsGrowth                           0.0090
grossMargins                             0.3296
operatingMargins                         0.1925
profitMargins                            0.1540


## Comparando tickers

In [6]:
tickers = ['WEGE3.SA', 'PETR4.SA', 'ITUB4.SA']

print(f'{"Ticker":<12} {"Preço":>8}  {"P/L":>7}  {"ROE":>8}  {"DY":>8}')
print('-' * 55)

for t in tickers:
    i = yf.Ticker(t).info
    preco = i.get('currentPrice', 0)
    pl    = i.get('trailingPE')
    roe   = i.get('returnOnEquity')
    dy    = i.get('dividendYield')

    print(
        f'{t:<12}'
        f' {f"R$ {preco:.2f}":>8}'
        f' {f"{pl:.1f}x" if pl else "N/A":>7}'
        f' {f"{roe*100:.1f}%" if roe else "N/A":>8}'
        f' {f"{dy*100:.2f}%" if dy else "N/A":>8}'
    )

Ticker          Preço      P/L       ROE        DY
-------------------------------------------------------
WEGE3.SA     R$ 48.19   32.3x    29.9%  322.00%


PETR4.SA     R$ 45.36    4.3x    30.3%  808.00%


ITUB4.SA     R$ 40.53    9.6x    21.0%  220.00%


In [7]:
#TODO adicionar isso na tool
#industry
#longBusinessSummary

## O que virou no projeto

A exploração acima virou a tool `buscar_dados_acao_b3` (`fluxo_financeiro/tools/financial_data.py`). P/L, ROE,
DY e os últimos 5 fechamentos continuam vindo do yfinance. A cotação, o nome, a faixa de 52 semanas, o valor de
mercado, o LPA (lucro por ação), o setor e o subsetor da B3, os maiores do setor e o sumário em português vêm da
Brapi (notebook 14), com o yfinance de reserva se ela não responder. Além disso, a tool:

- **corrige o DY** quando o yfinance já devolve em percentual (valor acima de 0,20 é dividido por 100: um DY de 88% não existe, era 0,88%);
- põe o sumário do negócio dentro de `<conteudo_externo>`: é texto escrito por terceiros, e o agente precisa saber que é dado, não instrução.

In [8]:
from fluxo_financeiro.tools.financial_data import buscar_dados_acao_b3

print(buscar_dados_acao_b3.invoke("WEGE3"))

2026-11-08 13:21:22,696 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=WEGE3.SA


2026-11-08 13:21:24,281 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ok (2443 chars)


Dados para WEGE3.SA:
Empresa: WEG SA
Preço Atual: R$ 48.19 (no dia: -0.23%)
Faixa de 52 semanas: R$ 41.15 a R$ 54.53
Valor de mercado: R$ 210.0 bi
LPA: R$ 1.49
P/L:  32.3x
ROE:  29.9%
DY:   3.22%
Setor: Bens Industriais | Subsetor: Máquinas e Equipamentos
Maiores do setor na Brapi (Producer Manufacturing, por valor de mercado): WEGE3 é a 1ª de 27.
  1. WEGE3 WEG S.A.: R$ 210.0 bi
  2. FRAS3 FRASLE MOBILITY S.A.: R$ 5.9 bi
  3. SHUL4 SCHULZ S.A.: R$ 5.4 bi
  4. DXCO3 DEXCO S.A.: R$ 4.3 bi
  5. POMO4 MARCOPOLO S.A.: R$ 4.3 bi
Sumario de Negócio:
<conteudo_externo origem="brapi">
A WEG S.A. é uma companhia brasileira de equipamentos eletroeletrônicos fundada em 1961, com sede em Jaraguá do Sul (SC), e listada na B3 sob o ticker WEGE3. A empresa atua no segmento de bens de capital industriais e desenvolve, fabrica e comercializa motores elétricos, geradores, transformadores, sistemas de automação, produtos para eletrificação e tintas industriais.

A estrutura operacional da companhia é org

## O que observar

- Os mesmos três números do protótipo aparecem já formatados: P/L com "x", ROE e DY em percentual.
- O sumário vem entre as tags `<conteudo_externo origem="brapi">`, em português. Se a Brapi não responder, ele vem do yfinance, em inglês, com `origem="yfinance"`.
- A última linha ("Fonte: ...") diz de onde veio cada parte.